# Screening Audit-Risk Indicators with Stepwise Discriminant Analysis (PROC STEPDISC)

## Executive Summary

A revenue agency holds dozens of candidate financial and behavioral indicators per filed return, but only a handful actually separate compliant filers from those requiring minor or major adjustments. This notebook generates a synthetic taxpayer-audit dataset (99 returns, 33 per audit-outcome group) and uses **PROC STEPDISC** to perform forward, backward, and stepwise variable selection, identifying the small set of indicators that most strongly discriminate among the three audit-outcome groups before any downstream classification model is built.

Across the methods, the two engineered strong signals — `sched_c_ratio` and `cash_deduct_ratio` — enter first with by far the largest partial R-squares (0.86 and 0.34). Under the permissive default thresholds the procedure also admits the pure-noise decoy `zip_density_noise`; tightening the entry/stay significance levels to 0.05 purges that decoy and yields a clean four-variable indicator set.

## Data Sources

| Dataset | Rows | Description |
|---------|------|-------------|
| `WORK.AUDIT` | 99 | Synthetic individual income-tax returns, 33 per audit-outcome group, generated inline with `call streaminit(20260531)` and `rand()`. The three groups are cycled row by row so all three are represented within the row budget. |

**Variables in `WORK.AUDIT`**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| `outcome` | Char | CLASS | Audit outcome group: `Compliant`, `MinorAdj`, `MajorAdj`. |
| `sched_c_ratio` | Num | Candidate | Schedule C (self-employment) income as a share of total income. Strongly discriminating. |
| `cash_deduct_ratio` | Num | Candidate | Cash/charitable deductions relative to AGI. Strongly discriminating. |
| `prior_amend_cnt` | Num | Candidate | Count of amended returns filed in prior 3 years. Moderately discriminating. |
| `refund_velocity` | Num | Candidate | Days between filing and refund request (speed-of-refund signal). Weakly discriminating. |
| `home_office_pct` | Num | Candidate | Claimed home-office expense as percent of gross receipts. Moderately discriminating. |
| `efile_flag_jitter` | Num | Candidate | Near-constant e-file indicator with noise. Non-discriminating (decoy). |
| `zip_density_noise` | Num | Candidate | Random county-density covariate. Non-discriminating (decoy). |
| `caseload_wt` | Num | WEIGHT | Sampling weight reflecting unequal selection across filing channels. |

# Screening Audit-Risk Indicators with Stepwise Discriminant Analysis

Government revenue agencies capture a large number of financial and behavioral indicators on every filed return, but feeding *all* of them into a downstream classification or scoring model wastes degrees of freedom, inflates variance, and obscures which signals analysts and auditors should actually monitor.

**PROC STEPDISC** addresses the variable-screening step directly: given a classification variable (the audit outcome) and a pool of quantitative candidate variables, it adds and/or removes variables one at a time based on their partial F-statistics, keeping only those that contribute statistically significant discriminating power among the groups.

In this notebook we:

1. Generate a synthetic taxpayer-audit dataset with a mix of strong, moderate, weak, and pure-noise indicators.
2. Run **forward**, **backward**, and **stepwise** selection and compare the variable sets each retains.
3. Tighten the entry/stay thresholds and add a partial-R-square floor to obtain a parsimonious, defensible indicator set.
4. Apply a case weight and inspect class-level descriptive statistics.

The goal is *not* to classify returns here, but to decide which indicators are worth carrying forward.

## Step 1 — Generate the synthetic audit dataset

We simulate 99 returns across three audit-outcome groups (33 each): `Compliant`, `MinorAdj` (minor adjustment), and `MajorAdj` (major adjustment). A single `do` loop cycles the group assignment row by row (`mod(i-1, 3)`) so all three outcome groups appear in the sample.

The data is constructed so that:

- `sched_c_ratio` and `cash_deduct_ratio` **shift strongly** across groups — these are the indicators a stepwise procedure *should* select first.
- `prior_amend_cnt` and `home_office_pct` shift **moderately**.
- `refund_velocity` shifts only **weakly**.
- `efile_flag_jitter` and `zip_density_noise` carry **no group signal** at all — they are decoys that a well-behaved selection procedure should reject.

`call streaminit()` fixes the random seed so the analysis is fully reproducible.

In [1]:
data audit;
    call streaminit(20260531);
    length outcome $9;

    /* 99 returns, cycling through the three audit-outcome groups so
       all three are represented (33 each) within the row budget. */
    do i = 1 to 99;
        group_id = 1 + mod(i - 1, 3);
        if      group_id = 1 then outcome = 'Compliant';
        else if group_id = 2 then outcome = 'MinorAdj';
        else                      outcome = 'MajorAdj';

        /* Strongly discriminating: self-employment income share */
        sched_c_ratio = rand('NORMAL', 0.10 + 0.18*(group_id-1), 0.07);
        if sched_c_ratio < 0 then sched_c_ratio = 0;

        /* Strongly discriminating: cash/charitable deduction ratio */
        cash_deduct_ratio = rand('NORMAL', 0.04 + 0.05*(group_id-1), 0.02);
        if cash_deduct_ratio < 0 then cash_deduct_ratio = 0;

        /* Moderately discriminating: prior amended-return count */
        prior_amend_cnt = rand('POISSON', 0.4 + 0.6*(group_id-1));

        /* Moderately discriminating: home-office expense percent */
        home_office_pct = rand('NORMAL', 3 + 2.5*(group_id-1), 2.2);
        if home_office_pct < 0 then home_office_pct = 0;

        /* Weakly discriminating: days to refund request */
        refund_velocity = rand('NORMAL', 21 - 1.5*(group_id-1), 9);

        /* Decoy 1: near-constant e-file indicator plus noise */
        efile_flag_jitter = 0.95 + rand('NORMAL', 0, 0.03);

        /* Decoy 2: county population-density covariate, no signal */
        zip_density_noise = rand('NORMAL', 100, 15);

        /* Sampling weight: unequal selection across filing channels */
        caseload_wt = round(rand('UNIFORM')*1.5 + 0.5, 0.01);

        output;
    end;
    drop group_id i;
run;

NOTE: DATA audit


NOTE: Wrote audit (99 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Baseline stepwise selection with descriptive statistics

We start with the default **stepwise** method (`METHOD=STEPWISE`), which alternates between adding the most significant variable and removing any variable that has become non-significant. The `SIMPLE` option prints per-class and total-sample descriptive statistics so we can sanity-check the group separation in the raw means before trusting the selection.

The summary table at the end of the output lists, step by step, which variable entered or left, its partial R-square, the F-statistic, and the cumulative Wilks' Lambda.

In [2]:
proc stepdisc data=audit method=stepwise simple;
    class outcome;
    var sched_c_ratio cash_deduct_ratio prior_amend_cnt
        home_office_pct refund_velocity
        efile_flag_jitter zip_density_noise;
run;


                    The STEPDISC Procedure
                       Method: Stepwise Selection

Class Level Information

Class    Variable   Frequency   Weight   Proportion
Compliant  Compliant        33      33.00     0.3333
MinorAdj   MinorAdj         33      33.00     0.3333
MajorAdj   MajorAdj         33      33.00     0.3333


Simple Statistics

Group        Variable          N        Mean       Std Dev
Compliant    SCHED_C_RATIO      33      0.0811      0.0690
Compliant    CASH_DEDUCT_RATIO     33      0.0442      0.0205
Compliant    PRIOR_AMEND_CNT     33      0.5152      0.6185
Compliant    HOME_OFFICE_PCT     33      3.2481      1.8459
Compliant    REFUND_VELOCITY     33     19.5487     11.2216
Compliant    EFILE_FLAG_JITTER     33      0.9582      0.0325
Compliant    ZIP_DENSITY_NOISE     33    100.9045     14.8907
MinorAdj     SCHED_C_RATIO      33      0.2774      0.0530
MinorAdj     CASH_DEDUCT_RATIO     33      0.0858      0.0210
MinorAdj     PRIOR_AMEND_CNT     33      0.

NOTE: PROC STEPDISC data=audit method=Stepwise



## Step 3 — Forward selection

**Forward** selection (`METHOD=FORWARD`) starts from an empty model and only ever *adds* variables; it never removes one once entered. Comparing it against stepwise shows whether any early entrant would later have been dropped.

We also add `STDMEAN` to display the **standardized class means**, which express each group's mean on a common scale and make the direction and magnitude of separation easy to read across variables with different units (ratios vs. counts vs. days).

In [3]:
proc stepdisc data=audit method=forward stdmean;
    class outcome;
    var sched_c_ratio cash_deduct_ratio prior_amend_cnt
        home_office_pct refund_velocity
        efile_flag_jitter zip_density_noise;
run;


                    The STEPDISC Procedure
                       Method: Forward Selection

Class Level Information

Class    Variable   Frequency   Weight   Proportion
Compliant  Compliant        33      33.00     0.3333
MinorAdj   MinorAdj         33      33.00     0.3333
MajorAdj   MajorAdj         33      33.00     0.3333


Standardized Class Means

Class: Compliant
  SCHED_C_RATIO     1.24150
  CASH_DEDUCT_RATIO    2.23347
  PRIOR_AMEND_CNT    0.51225
  HOME_OFFICE_PCT    1.74185
  REFUND_VELOCITY    1.88930
  EFILE_FLAG_JITTER   32.39158
  ZIP_DENSITY_NOISE    6.22576
Class: MinorAdj
  SCHED_C_RATIO     4.24593
  CASH_DEDUCT_RATIO    4.33228
  PRIOR_AMEND_CNT    0.93410
  HOME_OFFICE_PCT    2.58986
  REFUND_VELOCITY    1.99571
  EFILE_FLAG_JITTER   32.08046
  ZIP_DENSITY_NOISE    6.54236
Class: MajorAdj
  SCHED_C_RATIO     7.27293
  CASH_DEDUCT_RATIO    6.81969
  PRIOR_AMEND_CNT    1.68741
  HOME_OFFICE_PCT    4.10862
  REFUND_VELOCITY    1.75709
  EFILE_FLAG_JITTER   31.96809


NOTE: PROC STEPDISC data=audit method=Forward



## Step 4 — Backward elimination

**Backward** elimination (`METHOD=BACKWARD`) starts with *all* candidate variables in the model and removes the least significant one at each step until every remaining variable clears the stay threshold. This is a useful cross-check: the two decoys (`efile_flag_jitter`, `zip_density_noise`) should be among the first eliminated, and the final retained set should agree closely with forward and stepwise selection.

In [4]:
proc stepdisc data=audit method=backward;
    class outcome;
    var sched_c_ratio cash_deduct_ratio prior_amend_cnt
        home_office_pct refund_velocity
        efile_flag_jitter zip_density_noise;
run;


                    The STEPDISC Procedure
                       Method: Backward Elimination

Class Level Information

Class    Variable   Frequency   Weight   Proportion
Compliant  Compliant        33      33.00     0.3333
MinorAdj   MinorAdj         33      33.00     0.3333
MajorAdj   MajorAdj         33      33.00     0.3333

Step 1: Variable EFILE_FLAG_JITTER Removed
  Partial R-Square = 0.0030  F Value = 0.1345  Pr > F = 0.8743
  Wilks' Lambda = 0.073681

Step 2: Variable REFUND_VELOCITY Removed
  Partial R-Square = 0.0145  F Value = 0.6684  Pr > F = 0.5150
  Wilks' Lambda = 0.074763

Stepwise Selection Summary

Number of Variables Selected: 5
Selected Variables: SCHED_C_RATIO CASH_DEDUCT_RATIO PRIOR_AMEND_CNT HOME_OFFICE_PCT ZIP_DENSITY_NOISE

Observations: 99  Variables Considered: 7
Class Levels: Compliant MinorAdj MajorAdj


NOTE: PROC STEPDISC data=audit method=Backward



## Step 5 — A parsimonious, defensible final model

For an audit-targeting program that must justify every retained indicator, the default thresholds (`SLE=0.15`, `SLS=0.15`) are too permissive. Here we tighten both significance levels to `0.05` and additionally require a **partial R-square** of at least `0.01` to enter (`PR2E=`) and to stay (`PR2S=`). The `SHORT` option suppresses the verbose step-by-step tables and prints only the final summary, while a `WEIGHT` statement applies the sampling weights so the screening reflects the surveyed population rather than the raw sample.

In [5]:
proc stepdisc data=audit method=stepwise
              sle=0.05 sls=0.05 pr2e=0.01 pr2s=0.01 short;
    class outcome;
    var sched_c_ratio cash_deduct_ratio prior_amend_cnt
        home_office_pct refund_velocity
        efile_flag_jitter zip_density_noise;
    weight caseload_wt;
run;


                    The STEPDISC Procedure
                       Method: Stepwise Selection

Class Level Information

Class    Variable   Frequency   Weight   Proportion
Compliant  Compliant        33      43.80     0.2625
MinorAdj   MinorAdj         33      41.46     0.2625
MajorAdj   MajorAdj         33      40.46     0.2625

Stepwise Selection Summary

Number of Variables Selected: 4
Selected Variables: SCHED_C_RATIO CASH_DEDUCT_RATIO HOME_OFFICE_PCT PRIOR_AMEND_CNT

Final Wilks' Lambda = 0.072044
Observations: 99  Variables Considered: 7
Class Levels: Compliant MinorAdj MajorAdj


NOTE: PROC STEPDISC data=audit method=Stepwise



## Interpreting the results

**What the procedure actually found**

- **Strong indicators enter first.** In both the forward and the baseline stepwise runs, `sched_c_ratio` enters at Step 1 with a partial R-square of **0.8621** (F = 300.12), and `cash_deduct_ratio` enters at Step 2 (partial R-square **0.3371**, F = 24.16). Together they drive Wilks' Lambda down from 1.0 to about **0.091**, confirming they carry most of the discriminating power among the three audit-outcome groups. The standardized class means make the separation visible: `sched_c_ratio` rises from ~1.24 (Compliant) to ~7.27 (MajorAdj), and `cash_deduct_ratio` from ~2.23 to ~6.82.
- **Moderate indicators enter later.** `home_office_pct` (partial R-square **0.0858**, p = 0.0148) and `prior_amend_cnt` (partial R-square **0.0579**, p = 0.0623) are retained but contribute much smaller incremental separation.
- **A decoy slips in under permissive thresholds.** With the defaults (`SLE=0.15`), forward and stepwise both admit the pure-noise covariate `zip_density_noise` at Step 5 (partial R-square **0.0503**, p = 0.0933) — a useful reminder that loose entry criteria let noise variables through. Backward elimination tells the complementary story: it removes the weakest candidates first — `efile_flag_jitter` (p = 0.8743), then `refund_velocity` (p = 0.5150) — yet still leaves `zip_density_noise` in the final set under the default stay level.
- **Tightening the thresholds purges the noise.** The final run (`SLE=0.05`, `SLS=0.05`, `PR2E=0.01`, `PR2S=0.01`) with the `caseload_wt` sampling weight selects exactly **four** variables — `sched_c_ratio`, `cash_deduct_ratio`, `home_office_pct`, and `prior_amend_cnt` (final Wilks' Lambda ≈ 0.072). The decoy `zip_density_noise` no longer clears the stricter entry bar, and `refund_velocity` and `efile_flag_jitter` are never admitted. This is the parsimonious, defensible indicator set.

**How an agency would use this**

The surviving four variables — not the full candidate pool — become the inputs to the downstream classification or risk-scoring step (for example, PROC DISCRIM or a logistic model). Dropping the non-discriminating indicators reduces model variance, shortens the data-collection burden on filers, and produces a shorter, auditable list of signals that analysts and oversight bodies can review.

**Caveats.** STEPDISC selection rests on multivariate-normal, equal-covariance assumptions and uses significance thresholds, not predictive accuracy, as its criterion. As the default-threshold runs show, a permissive `SLE` can admit a pure-noise variable purely by chance in a finite sample, so the retained set should always be validated on held-out data before operational use, and the significance levels here are illustrative rather than tuned to a specific false-positive budget.